In [0]:
SELECT 
  c.year,
  c.month,
  p.payer_name,

  COUNT(*) as total_encounters,

  -- Zero coverage
  SUM(
    CASE 
      WHEN f.payer_id IS NULL OR f.total_claim_cost = 0 
      THEN 1 ELSE 0 
    END
  ) as zero_coverage,

  -- % zero coverage
  ROUND(
    SUM(CASE 
          WHEN f.payer_id IS NULL OR f.total_claim_cost = 0 
          THEN 1 ELSE 0 
        END) * 100.0 / COUNT(*),
    2
  ) as zero_coverage_pct,

  -- % covered
  ROUND(
    SUM(CASE 
          WHEN f.total_claim_cost > 0 
          THEN 1 ELSE 0 
        END) * 100.0 / COUNT(*),
    2
  ) as coverage_pct

FROM medical_catalog.gold.fact f
JOIN medical_catalog.gold.dim_calendar c 
  ON DATE(f.start_time) = c.date
LEFT JOIN medical_catalog.gold.dim_payer p 
  ON f.payer_id = p.payer_id

GROUP BY c.year, c.month, p.payer_name
ORDER BY c.year, c.month;
